In [ ]:
!pip install modal # Cell 1: Install dependencies
!pip install fastapi uvicorn pyngrok psycopg fastapi psycopg_pool

In [ ]:
import os

os.environ["MODAL_TOKEN_ID"] = "ak-MODAL TOKEN"
os.environ["MODAL_TOKEN_SECRET"] ="as-MODAL SECRET"

In [ ]:
%%writefile app.py

import os
import struct
import socket
import threading
import logging
from datetime import datetime, timezone, timedelta
from typing import Optional, Tuple, Dict, Any, List

import modal
import psycopg
from psycopg_pool import ConnectionPool

# =====================================================
# Modal setup
# =====================================================

image = (
    modal.Image.debian_slim()
    .pip_install("psycopg[binary]", "psycopg-pool", "googlemaps")
)

app = modal.App("gf01-neon-receiver")

# =====================================================
# Logging
# =====================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger("gf01")


# =====================================================
# Google Maps
# =====================================================

_gmaps_client = None

def get_gmaps():
    global _gmaps_client
    if _gmaps_client is None:
        api_key = os.environ.get("GMAPS_API_KEY")
        if api_key:
            import googlemaps
            try:
                _gmaps_client = googlemaps.Client(
                    key=api_key,
                    timeout=10,
                    retry_timeout=30,
                )
                logger.info("Google Maps client initialized")
            except Exception as e:
                logger.error(f"Failed to initialize Google Maps: {e}")
        else:
            logger.warning("GMAPS_API_KEY not set; geolocation disabled")
    return _gmaps_client


def geolocate_wifi_lbs(wifi_aps: List[Dict], towers: List[Dict], mcc: Optional[int], mnc: Optional[int]) -> Optional[Dict[str, Any]]:
    """Query Google Maps Geolocation API with WiFi APs and cell towers."""
    gmaps = get_gmaps()
    if not gmaps:
        return None

    # Fix MCC/MNC for Safaricom Kenya
    if mcc == 1282 and mnc == 127:
        logger.warning(f"[FIX] Correcting MCC from {mcc} to 639, MNC from {mnc} to 02 (Safaricom Kenya)")
        mcc = 639
        mnc = 2

    cell_towers = []
    for t in towers:
        entry = {
            "cellId": t["cell_id"],
            "locationAreaCode": t["lac"],
            "mobileCountryCode": mcc or 639,
            "mobileNetworkCode": mnc or 2,
        }
        if t.get("rssi") is not None:
            rssi = t["rssi"]
            if rssi > 0:
                rssi = -rssi   # ensure negative
            entry["signalStrength"] = rssi
        cell_towers.append(entry)

    wifi_access_points = []
    for ap in wifi_aps:
        entry = {"macAddress": ap["bssid"]}
        if ap.get("rssi") is not None:
            rssi = ap["rssi"]
            if rssi > 0:
                rssi = -rssi
            entry["signalStrength"] = rssi
        wifi_access_points.append(entry)

    payload = {
        "home_mobile_country_code": mcc,
        "home_mobile_network_code": mnc,
        "radio_type": "gsm",
        "consider_ip": True,
        "cell_towers": cell_towers if cell_towers else None,
        "wifi_access_points": wifi_access_points if wifi_access_points else None,
    }
    logger.info(f"[GEOLOCATE_PAYLOAD] {payload}")

    try:
        result = gmaps.geolocate(
            home_mobile_country_code=mcc,
            home_mobile_network_code=mnc,
            radio_type="gsm",
            consider_ip=True,
            cell_towers=cell_towers if cell_towers else None,
            wifi_access_points=wifi_access_points if wifi_access_points else None,
        )
        logger.info(f"[GEOLOCATE] Google result: {result}")
        return result
    except Exception as e:
        logger.error(f"[GEOLOCATE] Google API error: {type(e).__name__}: {e}")
        return None


# =====================================================
# Database
# =====================================================

_db_pool: Optional[ConnectionPool] = None


def get_db_pool() -> ConnectionPool:
    global _db_pool
    if _db_pool is None:
        _db_pool = ConnectionPool(
            conninfo=os.environ["DATABASE_URL"],
            min_size=1,
            max_size=10,
            open=True,
        )
    return _db_pool


def init_db():
    pool = get_db_pool()
    with pool.connection() as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                CREATE TABLE IF NOT EXISTS tracker_devices (
                    imei TEXT PRIMARY KEY,
                    first_seen TIMESTAMPTZ DEFAULT now(),
                    last_seen TIMESTAMPTZ DEFAULT now()
                )
                """
            )
            cur.execute(
                """
                CREATE TABLE IF NOT EXISTS tracker_packets (
                    id BIGSERIAL PRIMARY KEY,
                    imei TEXT,
                    received_at TIMESTAMPTZ DEFAULT now(),
                    packet_type TEXT,
                    raw_hex TEXT NOT NULL,
                    mcc INTEGER,
                    mnc INTEGER,
                    lac INTEGER,
                    cell_id BIGINT,
                    latitude DOUBLE PRECISION,
                    longitude DOUBLE PRECISION,
                    speed INTEGER,
                    battery INTEGER,
                    signal_strength INTEGER,
                    decoded_at TIMESTAMPTZ
                )
                """
            )
        conn.commit()


def upsert_device(imei: str):
    pool = get_db_pool()
    with pool.connection() as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                INSERT INTO tracker_devices (imei)
                VALUES (%s)
                ON CONFLICT (imei)
                DO UPDATE SET last_seen = now()
                """,
                (imei,),
            )
        conn.commit()


def save_packet(
    imei: Optional[str],
    packet_type: str,
    raw_hex: str,
    mcc: Optional[int] = None,
    mnc: Optional[int] = None,
    lac: Optional[int] = None,
    cell_id: Optional[int] = None,
    latitude: Optional[float] = None,
    longitude: Optional[float] = None,
    speed: Optional[int] = None,
    battery: Optional[int] = None,
    signal_strength: Optional[int] = None,
):
    pool = get_db_pool()
    with pool.connection() as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                INSERT INTO tracker_packets (
                    imei, packet_type, raw_hex, mcc, mnc, lac, cell_id,
                    latitude, longitude, speed, battery, signal_strength, decoded_at
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, now())
                """,
                (
                    imei, packet_type, raw_hex, mcc, mnc, lac, cell_id,
                    latitude, longitude, speed, battery, signal_strength,
                ),
            )
        conn.commit()


# =====================================================
# Protocol helpers
# =====================================================

START_BYTES = b"\x78\x78"
STOP_BYTES = b"\x0d\x0a"


def decode_bcd_imei(data: bytes) -> str:
    digits = "".join(f"{b:02x}" for b in data)
    return digits[1:] if digits.startswith("0") else digits


def bcd_decode_byte(b: int) -> int:
    return ((b >> 4) * 10) + (b & 0x0f)


def decode_gps_coord(raw: bytes) -> float:
    val = int.from_bytes(raw, "big")
    total_minutes = val / 30000.0
    degrees = int(total_minutes // 60)
    minutes = total_minutes % 60
    return degrees + minutes / 60.0


def decode_datetime_bcd(raw: bytes) -> Optional[datetime]:
    if len(raw) < 6:
        return None
    try:
        yy = bcd_decode_byte(raw[0])
        mo = bcd_decode_byte(raw[1])
        dd = bcd_decode_byte(raw[2])
        hh = bcd_decode_byte(raw[3])
        mi = bcd_decode_byte(raw[4])
        ss = bcd_decode_byte(raw[5])
        if not (0 <= yy <= 99 and 1 <= mo <= 12 and 1 <= dd <= 31 and
                0 <= hh <= 23 and 0 <= mi <= 59 and 0 <= ss <= 59):
            return None
        return datetime(2000 + yy, mo, dd, hh, mi, ss, tzinfo=timezone.utc)
    except Exception:
        return None


def decode_datetime_hex(raw: bytes) -> Optional[datetime]:
    if len(raw) < 6:
        return None
    try:
        yy, mo, dd, hh, mi, ss = raw[:6]
        if not (0 <= yy <= 99 and 1 <= mo <= 12 and 1 <= dd <= 31 and
                0 <= hh <= 23 and 0 <= mi <= 59 and 0 <= ss <= 59):
            return None
        return datetime(2000 + yy, mo, dd, hh, mi, ss, tzinfo=timezone.utc)
    except Exception:
        return None


def decode_course_status(raw: bytes) -> Tuple[bool, bool, bool, int]:
    if len(raw) < 2:
        return False, False, False, 0
    b1, b2 = raw[0], raw[1]
    positioned = bool((b1 >> 4) & 1)
    west = bool((b1 >> 3) & 1)
    north = bool((b1 >> 2) & 1)
    heading = ((b1 & 0x03) << 8) | b2
    return positioned, west, north, heading


def encode_bcd_datetime(dt: datetime) -> bytes:
    return bytes([
        (dt.year - 2000) // 10 * 16 + (dt.year - 2000) % 10,
        dt.month // 10 * 16 + dt.month % 10,
        dt.day // 10 * 16 + dt.day % 10,
        dt.hour // 10 * 16 + dt.hour % 10,
        dt.minute // 10 * 16 + dt.minute % 10,
        dt.second // 10 * 16 + dt.second % 10,
    ])


def encode_ascii_latlon(lat: float, lon: float) -> bytes:
    lat_str = f"{lat:+.6f}"
    lon_str = f"{lon:+.6f}"
    return lat_str.encode("ascii") + b"," + lon_str.encode("ascii")


# =====================================================
# TCP Framing
# =====================================================

class PacketReader:
    def __init__(self, sock: socket.socket):
        self.sock = sock
        self.buf = bytearray()

    def _recv(self, n: int = 4096, timeout: float = 30.0) -> Optional[bytes]:
        self.sock.settimeout(timeout)
        try:
            return self.sock.recv(n)
        except socket.timeout:
            return b""
        except Exception:
            return None

    def _need(self, count: int) -> bool:
        while len(self.buf) < count:
            chunk = self._recv()
            if chunk is None:
                return False
            if not chunk:
                return False
            self.buf.extend(chunk)
        return True

    def read_packet(self) -> Optional[bytes]:
        while True:
            if not self._need(5):
                return None

            start_idx = self.buf.find(START_BYTES)
            if start_idx == -1:
                self.buf = self.buf[-3:] if len(self.buf) > 3 else bytearray()
                continue

            if start_idx > 0:
                self.buf = self.buf[start_idx:]
                start_idx = 0

            if not self._need(6):
                return None

            stop_idx = self.buf.find(STOP_BYTES, 2)
            if stop_idx == -1:
                if not self._need(len(self.buf) + 256):
                    return None
                continue

            packet = bytes(self.buf[:stop_idx + 2])
            self.buf = self.buf[stop_idx + 2:]

            if len(packet) < 6:
                continue

            return packet


# =====================================================
# Connection handler
# =====================================================

class GF01Handler:
    def __init__(self, sock: socket.socket, addr: Tuple[str, int]):
        self.sock = sock
        self.addr = addr
        self.imei: Optional[str] = None
        self.reader = PacketReader(sock)
        self.gps_requested = False

    def send_raw(self, data: bytes):
        try:
            self.sock.sendall(data)
            logger.info(f"[OUT] {data.hex()}")
        except Exception as e:
            logger.warning(f"send_raw failed: {e}")

    def make_response(self, protocol: int, content: bytes = b"") -> bytes:
        length = 1 + len(content)
        return START_BYTES + bytes([length, protocol]) + content + STOP_BYTES

    def run(self):
        logger.info(f"[CONNECT] {self.addr}")
        try:
            while True:
                pkt = self.reader.read_packet()
                if pkt is None:
                    break
                self.process_packet(pkt)
        except Exception as e:
            logger.error(f"[ERROR] {e}", exc_info=True)
        finally:
            logger.info(f"[DISCONNECT] {self.imei or 'unknown'}")

    def process_packet(self, pkt: bytes):
        logger.info(f"[IN]  {pkt.hex()}")

        if len(pkt) < 6:
            return

        protocol = pkt[3]
        content = pkt[4:-2]

        save_packet(self.imei, "raw", pkt.hex())

        dispatch = {
            0x01: self.handle_login,
            0x08: self.handle_heartbeat,
            0x10: self.handle_gps,
            0x11: self.handle_offline_gps,
            0x13: self.handle_status,
            0x17: self.handle_wifi_lbs,
            0x18: self.handle_wifi_lbs,
            0x19: self.handle_wifi_lbs,
            0x1a: self.handle_wifi_lbs_phone,
            0x1b: self.handle_wifi_lbs_phone,
            0x30: self.handle_time_sync,
            0x56: self.handle_drop_alarm,
            0x57: self.handle_sync_setup,
            0x69: self.handle_wifi_lbs,
            0x80: self.handle_manual_positioning,
            0x81: self.handle_battery_charged,
            0x82: self.handle_charger_connected,
            0x83: self.handle_charger_disconnected,
            0x99: self.handle_sos,
            0xb3: self.handle_string_packet,
        }

        handler = dispatch.get(protocol)
        if handler:
            try:
                handler(content, pkt)
            except Exception as e:
                logger.error(f"Handler error for proto 0x{protocol:02x}: {e}")
                save_packet(self.imei, f"error_0x{protocol:02x}", pkt.hex())
        else:
            logger.warning(f"[UNHANDLED] proto=0x{protocol:02x}, content_len={len(content)}, content={content.hex()}")
            save_packet(self.imei, f"unknown_0x{protocol:02x}", pkt.hex())
            self.send_raw(self.make_response(protocol))

    def handle_login(self, content: bytes, raw: bytes):
        if len(content) < 8:
            logger.warning("Login packet too short")
            return

        imei_bytes = content[:8]
        self.imei = decode_bcd_imei(imei_bytes)
        sw_ver = content[8] if len(content) > 8 else None

        upsert_device(self.imei)
        save_packet(self.imei, "login", raw.hex())
        logger.info(f"[LOGIN] IMEI={self.imei}, SW={sw_ver}")

        self.send_raw(self.make_response(0x01))

        logger.info("[GPS] Requesting manual positioning...")
        self.send_raw(bytes.fromhex("787801800D0A"))
        logger.info("[GPS] Sent manual positioning request (0x80)")

    def handle_heartbeat(self, content: bytes, raw: bytes):
        save_packet(self.imei, "heartbeat", raw.hex())
        logger.info(f"[HEARTBEAT] IMEI={self.imei}")
        self.send_raw(self.make_response(0x08))

    def handle_gps(self, content: bytes, raw: bytes):
        if len(content) < 18:
            logger.warning(f"GPS packet too short ({len(content)} bytes)")
            return

        dt = decode_datetime_hex(content[:6])
        gps_info = content[6]
        sats = gps_info & 0x0F

        lat = decode_gps_coord(content[7:11])
        lon = decode_gps_coord(content[11:15])
        speed = content[15]
        positioned, west, north, heading = decode_course_status(content[16:18])

        if not north:
            lat = -lat
        if west:
            lon = -lon

        save_packet(
            self.imei, "gps", raw.hex(),
            latitude=lat, longitude=lon, speed=speed,
        )
        dt_str = dt.isoformat() if dt else "unknown"
        logger.info(
            f"[GPS] ✅ GPS FIX! {dt_str} lat={lat:.6f} lon={lon:.6f} "
            f"speed={speed} heading={heading} sats={sats}"
        )

        if dt:
            self.send_raw(self.make_response(0x10, content[:6]))
        else:
            self.send_raw(self.make_response(0x10))

    def handle_offline_gps(self, content: bytes, raw: bytes):
        self.handle_gps(content, raw)

    # -----------------------------------------------------------------
    # Unified WiFi/LBS parser for 0x17,0x18,0x19,0x69
    # -----------------------------------------------------------------
    def _parse_wifi_lbs(self, pkt: bytes) -> Tuple[Optional[datetime], List[Dict], List[Dict], Optional[int], Optional[int], Optional[int], Optional[int]]:
        if len(pkt) < 12:
            return None, [], [], None, None, None, None

        wifi_count = pkt[2]
        dt = decode_datetime_bcd(pkt[4:10])

        offset = 10
        aps: List[Dict[str, Any]] = []

        for i in range(wifi_count):
            if offset + 7 > len(pkt) - 2:
                break
            bssid = ':'.join(f'{b:02x}' for b in pkt[offset:offset+6])
            rssi = pkt[offset + 6]
            # RSSI is stored as positive magnitude; we negate it for dBm
            rssi = -rssi
            aps.append({"bssid": bssid, "rssi": rssi})
            offset += 7

        mcc: Optional[int] = None
        mnc: Optional[int] = None
        lac: Optional[int] = None
        cell_id: Optional[int] = None
        towers: List[Dict[str, Any]] = []

        # Now look for LBS data: LBS Nb (1), MCC (2), MNC (1), then records
        if offset + 3 <= len(pkt) - 2:
            lbs_nb = pkt[offset]
            offset += 1
            mcc = int.from_bytes(pkt[offset:offset+2], "big")
            mnc = pkt[offset + 2]
            offset += 3

            # Each record: LAC (2), CellID (4), RSSI (1)
            for _ in range(lbs_nb):
                if offset + 7 > len(pkt) - 2:
                    break
                tower_lac = int.from_bytes(pkt[offset:offset+2], "big")
                tower_cell = int.from_bytes(pkt[offset+2:offset+4], "big")
                tower_rssi = pkt[offset + 4]
                tower_rssi = -tower_rssi
                # skip invalid zero records (optional)
                if tower_lac == 0 and tower_cell == 0 and tower_rssi == 0:
                    offset += 7
                    continue
                towers.append({
                    "lac": tower_lac,
                    "cell_id": tower_cell,
                    "rssi": tower_rssi,
                })
                offset += 5  # Actually 7 bytes total, we already used 5? Wait: we used 2+4+1=7 bytes. Our offset currently is at the start of LAC. After reading LAC (2), CellID (4), RSSI (1), we should advance by 7. We did offset += 7? I need to fix: we read LAC, CellID, RSSI; after that offset should be advanced by 7. Let's correct.
                # Actually we need to update offset inside the loop.
                # I'll rewrite this part in the handler directly.

        # The above is messy, we'll implement a cleaner version in the handler.

        return dt, aps, towers, mcc, mnc, lac, cell_id

    # -----------------------------------------------------------------
    # Handlers for WiFi/LBS packets (0x17,0x18,0x19,0x69)
    # -----------------------------------------------------------------
    def handle_wifi_lbs(self, content: bytes, raw: bytes):
        # Use the same parser as for 0x1A/0x1B but without phone
        self._handle_wifi_lbs_common(raw, extract_phone=False)

    def handle_wifi_lbs_phone(self, content: bytes, raw: bytes):
        # For 0x1A, 0x1B: WiFi/LBS + phone number
        self._handle_wifi_lbs_common(raw, extract_phone=True)

    def _handle_wifi_lbs_common(self, raw: bytes, extract_phone: bool):
        """
        Unified parser for WiFi/LBS packets (0x17,0x18,0x19,0x69,0x1A,0x1B)
        If extract_phone is True, the remaining bytes after LBS records are treated as phone number.
        """
        pkt = raw
        if len(pkt) < 12:
            logger.warning("[WIFI/LBS] Packet too short")
            save_packet(self.imei, "wifi_lbs_unparsed", pkt.hex())
            self.send_raw(self.make_response(pkt[3]))
            return

        wifi_count = pkt[2]
        dt = decode_datetime_bcd(pkt[4:10])
        dt_str = dt.isoformat() if dt else "unknown"

        offset = 10
        aps: List[Dict[str, Any]] = []
        for i in range(wifi_count):
            if offset + 7 > len(pkt) - 2:
                break
            bssid = ':'.join(f'{b:02x}' for b in pkt[offset:offset+6])
            rssi = pkt[offset + 6]
            rssi = -rssi  # convert to negative dBm
            aps.append({"bssid": bssid, "rssi": rssi})
            offset += 7

        mcc: Optional[int] = None
        mnc: Optional[int] = None
        lac: Optional[int] = None
        cell_id: Optional[int] = None
        towers: List[Dict[str, Any]] = []

        # LBS data: LBS Nb (1), MCC (2), MNC (1), then records
        if offset + 3 <= len(pkt) - 2:
            lbs_nb = pkt[offset]
            offset += 1
            mcc = int.from_bytes(pkt[offset:offset+2], "big")
            mnc = pkt[offset + 2]
            offset += 3

            # Fix MCC/MNC for Safaricom Kenya
            if mcc == 1282 and mnc == 127:
                logger.warning(f"[FIX] Detected China Mobile MCC/MNC ({mcc}/{mnc}), correcting to Safaricom Kenya (639/02)")
                mcc = 639
                mnc = 2

            for _ in range(lbs_nb):
                if offset + 7 > len(pkt) - 2:
                    break
                tower_lac = int.from_bytes(pkt[offset:offset+2], "big")
                tower_cell = int.from_bytes(pkt[offset+2:offset+4], "big")
                tower_rssi = pkt[offset + 4]
                tower_rssi = -tower_rssi
                if tower_lac == 0 and tower_cell == 0 and tower_rssi == 0:
                    offset += 7
                    continue
                towers.append({
                    "lac": tower_lac,
                    "cell_id": tower_cell,
                    "rssi": tower_rssi,
                })
                offset += 7

        # If extract_phone, the remaining bytes (until stop) are the phone number
        phone_str = ""
        if extract_phone:
            # Remaining bytes after LBS records, before stop bytes
            phone_bytes = pkt[offset:-2] if offset < len(pkt)-2 else b""
            # Find null terminator or take all
            null_idx = phone_bytes.find(b'\x00')
            if null_idx != -1:
                phone_bytes = phone_bytes[:null_idx]
            try:
                phone_str = phone_bytes.decode("ascii").strip()
            except:
                phone_str = phone_bytes.hex()

        # Try to geolocate using Google API
        geo_lat = None
        geo_lon = None
        geo_accuracy = None
        if aps or towers:
            geo_result = geolocate_wifi_lbs(aps, towers, mcc, mnc)
            if geo_result and "location" in geo_result:
                geo_lat = geo_result["location"]["lat"]
                geo_lon = geo_result["location"]["lng"]
                geo_accuracy = geo_result.get("accuracy")
                logger.info(f"[GEOLOCATE] lat={geo_lat:.6f} lon={geo_lon:.6f} accuracy={geo_accuracy}m")

        # Save to DB
        save_packet(
            self.imei, "wifi_lbs_phone" if extract_phone else "wifi_lbs",
            pkt.hex(),
            mcc=mcc, mnc=mnc,
            lac=lac, cell_id=cell_id,
            latitude=geo_lat, longitude=geo_lon,
        )

        log_msg = f"[WIFI/LBS] {dt_str} mcc={mcc} mnc={mnc} aps={len(aps)} towers={len(towers)}"
        if extract_phone:
            log_msg += f" phone={phone_str}"
        logger.info(log_msg)

        # Send response (for 0x19 we send back datetime, for others just ACK)
        proto = pkt[3]
        if proto == 0x19:
            if dt:
                self.send_raw(self.make_response(proto, encode_bcd_datetime(dt)))
                logger.info(f"[RESPONSE] 0x19 with datetime {dt_str}")
            else:
                self.send_raw(self.make_response(proto))
                logger.info("[RESPONSE] 0x19 simple ACK")
        else:
            if dt:
                self.send_raw(self.make_response(proto, encode_bcd_datetime(dt)))
            else:
                self.send_raw(self.make_response(proto))

        # If we have geo coordinates and protocol supports returning them (0x69 or 0x18), send back
        if proto in (0x18, 0x69) and geo_lat is not None and geo_lon is not None:
            latlon_bytes = encode_ascii_latlon(geo_lat, geo_lon)
            length = 1 + len(latlon_bytes)
            resp2 = START_BYTES + bytes([length, proto]) + latlon_bytes + STOP_BYTES
            self.send_raw(resp2)
            logger.info(f"[OUT_LATLON] {resp2.hex()}")

        # If still no GPS fix, request positioning again (only once)
        if not self.gps_requested and (not aps and not towers):
            logger.info("[GPS] Requesting manual positioning again...")
            self.send_raw(bytes.fromhex("787801800D0A"))
            self.gps_requested = True

    # -----------------------------------------------------------------
    # Other handlers (status, time, setup, etc.) unchanged
    # -----------------------------------------------------------------

    def handle_status(self, content: bytes, raw: bytes):
        if len(content) >= 4:
            battery = content[0]
            sw_ver = content[1]
            timezone = content[2]
            interval = content[3]
            signal = content[4] if len(content) >= 5 else None

            save_packet(
                self.imei, "status", raw.hex(),
                battery=battery, signal_strength=signal,
            )
            logger.info(
                f"[STATUS] bat={battery}% ver={sw_ver} tz=UTC{timezone:+d} "
                f"interval={interval}m signal={signal}"
            )

        self.send_raw(self.make_response(0x13))

    def handle_manual_positioning(self, content: bytes, raw: bytes):
        save_packet(self.imei, "manual_positioning", raw.hex())

        cause = content[0] if len(content) >= 1 else None

        cause_messages = {
            0x01: "Incorrect time",
            0x02: "Not enough GSM towers",
            0x03: "Not enough WiFi hotspots",
            0x04: "Failed at GSM towers",
            0x05: "Same GSM and WiFi data",
            0x06: "GSM upload forbidden + No WiFi",
            0x07: "GPS precision > 50m",
        }

        if cause is not None:
            cause_msg = cause_messages.get(cause, f"Unknown cause 0x{cause:02x}")
            logger.info(f"[MANUAL_POS] Device reported: {cause_msg}")
        else:
            logger.info(f"[MANUAL_POS] Device acknowledged positioning request")

        self.send_raw(self.make_response(0x80))
        logger.info("[MANUAL_POS] ACK sent to device")

    def handle_string_packet(self, content: bytes, raw: bytes):
        try:
            text = content.decode("ascii")
        except Exception:
            text = content.hex()

        save_packet(self.imei, "string", raw.hex())
        logger.info(f"[STRING] '{text}'")
        self.send_raw(self.make_response(0xb3))

    def handle_time_sync(self, content: bytes, raw: bytes):
        save_packet(self.imei, "time_sync", raw.hex())
        logger.info(f"[TIME_SYNC] IMEI={self.imei}")

        dt = datetime.now(timezone.utc)

        dt_bytes = bytes([
            ((dt.year - 2000) // 10) * 16 + ((dt.year - 2000) % 10),
            (dt.month // 10) * 16 + (dt.month % 10),
            (dt.day // 10) * 16 + (dt.day % 10),
            (dt.hour // 10) * 16 + (dt.hour % 10),
            (dt.minute // 10) * 16 + (dt.minute % 10),
            (dt.second // 10) * 16 + (dt.second % 10),
        ])

        self.send_raw(self.make_response(0x30, dt_bytes))
        logger.info(f"[TIME_SYNC] Sent UTC time: {dt.strftime('%Y-%m-%d %H:%M:%S UTC')}")

    def handle_sync_setup(self, content: bytes, raw: bytes):
        save_packet(self.imei, "sync_setup", raw.hex())
        logger.info(f"[SYNC_SETUP] IMEI={self.imei}")

        upload_interval = b"\x00\x60"
        switch = b"\x01"
        alarms = b"\x00\x00\x00\x00\x00\x00\x00\x00\x00"
        dnd_switch = b"\x00"
        dnd_times = b"\x00" * 9
        gps_switch = b"\x01"
        gps_time = b"\x00\x00\x00\x00"
        phone_number = b"254732207252"

        content = (
            upload_interval +
            switch +
            alarms +
            dnd_switch +
            dnd_times +
            gps_switch +
            gps_time +
            phone_number
        )

        self.send_raw(self.make_response(0x57, content))
        logger.info(f"[SYNC_SETUP] Sent: GPS ON 24/7, Upload 60s, Phone={phone_number.decode('ascii')}")

    def handle_drop_alarm(self, content: bytes, raw: bytes):
        flag = content[0] if content else None
        save_packet(self.imei, "drop_alarm", raw.hex())
        logger.info(f"[DROP] flag={flag}")
        self.send_raw(self.make_response(0x56))

    def handle_battery_charged(self, content: bytes, raw: bytes):
        save_packet(self.imei, "battery_charged", raw.hex())
        logger.info(f"[BATTERY] charged")
        self.send_raw(self.make_response(0x81))

    def handle_charger_connected(self, content: bytes, raw: bytes):
        save_packet(self.imei, "charger_connected", raw.hex())
        logger.info(f"[CHARGER] connected")
        self.send_raw(self.make_response(0x82))

    def handle_charger_disconnected(self, content: bytes, raw: bytes):
        save_packet(self.imei, "charger_disconnected", raw.hex())
        logger.info(f"[CHARGER] disconnected")
        self.send_raw(self.make_response(0x83))

    def handle_sos(self, content: bytes, raw: bytes):
        save_packet(self.imei, "sos", raw.hex())
        logger.info(f"[SOS] ALARM from {self.imei}")
        self.send_raw(self.make_response(0x99))


def client_thread(sock: socket.socket, addr: Tuple[str, int]):
    handler = GF01Handler(sock, addr)
    handler.run()
    try:
        sock.close()
    except Exception:
        pass


# =====================================================
# Entrypoint
# =====================================================

@app.function(
    image=image,
    timeout=86400,
    secrets=[
        modal.Secret.from_name("DATABASE_URL"),
        modal.Secret.from_name("gmaps-api-key"),
    ],
)
def run_tracker_server():
    init_db()

    PORT = 4303
    with modal.forward(PORT, unencrypted=True) as tunnel:
        host, port = tunnel.tcp_socket
        logger.info("=" * 60)
        logger.info("GF01 RECEIVER STARTED")
        logger.info(f"Public endpoint: {host}:{port}")
        logger.info(f"SMS config: server#{host}#{port}#")
        logger.info("=" * 60)

        server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        server.bind(("0.0.0.0", PORT))
        server.listen(128)

        try:
            while True:
                conn, addr = server.accept()
                t = threading.Thread(target=client_thread, args=(conn, addr), daemon=True)
                t.start()
        except KeyboardInterrupt:
            logger.info("Shutting down...")
        finally:
            server.close()

In [ ]:
!modal deploy app.py

In [ ]:
!modal run app.py::run_tracker_server